<a href="https://colab.research.google.com/github/gigimcc4/IBM_for_researchers/blob/main/Copy_of_Day4_Simulating_Agentic_Loop_v3_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **IMPORTANT - FIRST - MAKE A COPY OF THIS NOTEBOOK, OR NOTHING WILL BE SAVED! FILE -> SAVE A COPY IN DRIVE/GITHUB**

# Day 4 · Simulating the Agentic Loop

*Created for the Nagoya University 5-Day Faculty Seminar on Teaching with AI*  
**Dr. Jeanne McClure, PhD** | NC State University Data Science Academy / Ars Innovate

---

### 📜 License

© 2026 Dr. Jeanne McClure. This work is licensed under a [Creative Commons Attribution-NonCommercial 4.0 International License (CC BY-NC 4.0)](https://creativecommons.org/licenses/by-nc/4.0/).

You are free to **share** (copy, redistribute) and **adapt** (remix, transform, build upon) this material, under the following terms:

- **Attribution** — You must give appropriate credit, provide a link to the license, and indicate if changes were made.
- **NonCommercial** — You may not use the material for commercial purposes.

This material was in part developed with support from the National Science Foundation, DUE-2313644. All opinions, findings, conclusions and recommendations expressed herein are those of the authors and do not necessarily reflect the views of the Foundation.

---

## ⚠️ What this notebook is — and what it is not

**What this IS:**
A simulation of agentic behavior using the same IBM Granite model from Tutorial 1.
We call the model three times in sequence, passing each output as input to the next step.
This makes the loop **visible and narrate-able.**

**What this is NOT:**
A true AI agent. In a real agentic system the model itself decides what step to take next,
which tools to call, and when to stop. That is what you will build in the
**Extension session** using IBM BeeAI — where the agent drives the loop, not the code.

---

**Before running:** Go to **Runtime → Change runtime type → T4 GPU**
Same as Tutorial 1 — Granite needs a GPU.

| Step | What happens |
|------|-------------|
| 0 · Setup | Check GPU + install libraries + load Granite from Hugging Face |
| 1 · Load data | Same dataset from yesterday |
| 2 · Plan | Model reads the data and writes a plan |
| 3 · Analyze | Model uses its own plan as input |
| 4 · Reflect | Model names the limits of its own output |

---
## STEP 0A · Check GPU

If this returns ❌ go to **Runtime → Change runtime type → T4 GPU → Save** then re-run.

In [ ]:
# Step 1: Check GPU availability
# This cell checks whether Colab has assigned us a GPU.
# If not, it tells you how to enable one.

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f'✅ GPU available: {gpu_info}')
    print(f'\n   You\'re all set! A T4 GPU has about 15 GB of memory,')
    print(f'   which is more than enough for our 2B parameter model.')
else:
    print('❌ No GPU detected!')
    print('\n   To fix this:')
    print('   1. Click "Runtime" in the menu bar above')
    print('   2. Click "Change runtime type"')
    print('   3. Under "Hardware accelerator," select "T4 GPU"')
    print('   4. Click "Save"')
    print('   5. Re-run this cell to confirm')

---
## STEP 0B · Install Libraries

In [ ]:
!pip install -q transformers accelerate torch sentencepiece protobuf
print('✓ Libraries ready')

---
## STEP 0C · Load IBM Granite from Hugging Face

Same model as Tutorial 1. Downloads once (~1.5 GB), then loads into GPU memory.
Takes 1–2 minutes. Wait for **✓ Granite ready**.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = 'ibm-granite/granite-3.1-2b-instruct'

print(f'Loading {MODEL_NAME} from Hugging Face...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto'
)
print('✓ Granite ready')

---
## STEP 0D · Helper Function

Same pattern as Tutorial 1 — one reusable function to call Granite.

In [ ]:
def ask_granite(system_prompt, user_prompt, label='', max_new_tokens=400):
    """Call Granite with a system prompt and user message. Print and return the response."""
    if label:
        print(f'\n{"="*60}')
        print(f'  {label}')
        print(f'{"="*60}\n')

    messages = [
        {'role': 'system',  'content': system_prompt},
        {'role': 'user',    'content': user_prompt}
    ]

    _encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt'
    )
    if hasattr(_encoded, 'input_ids'):
        input_ids = _encoded.input_ids.to(model.device)
    else:
        input_ids = _encoded.to(model.device)

    attention_mask = torch.ones_like(input_ids)

    output = model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode only the new tokens
    new_tokens = output[0][input_ids.shape[-1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(result)
    return result

print('✓ Helper function ready')

---
## STEP 1 · Load the Dataset

Same 943 students, 5 science courses from yesterday.

In [ ]:
import pandas as pd

url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSMMITWgpJn9BniKnTlH60oQUfc7UHJo22F9ZCt4A2OeOOBkspM15Y3yh8lNlwKKY2ay9LTZGUHO_jV/pub?gid=1964674133&single=true&output=csv'
df = pd.read_csv(url)

# Pre-compute stats to pass to the model
grade_by_subject = df.groupby('subject')['proportion_earned'].mean().mul(100).round(1)
time_by_subject  = df.groupby('subject')['time_spent_hours'].mean().round(1)

print(f'✓ {df.shape[0]} students · {df.shape[1]} columns')
print(f'\nGrade averages by subject (%):')
print(grade_by_subject.to_string())
print(f'\nAverage time on platform by subject (hours):')
print(time_by_subject.to_string())

---
## The ICI System Prompt

Read this before running the simulation. Find the three components:

| Component | What it does here |
|-----------|-------------------|
| **Instruction** | Defines the deliverable and the stop condition |
| **Context** | States what the model must NOT do |
| **Input** | States the escalation rule — when to stop and wait for a human |

In [ ]:
SYSTEM = """
You are a data analysis assistant working with a university student dataset.

INSTRUCTION (your goal and deliverable):
Analyze grade patterns across subjects.
Produce: a written plan, three key findings, and one recommendation
for what a human educator should examine more closely.
Stop after the recommendation. Do not produce code.

CONTEXT (what you must NOT do):
Do NOT recommend specific instructional interventions.
Do NOT draw conclusions about student ability or effort.
If a finding requires human interpretation, mark it:
[HUMAN JUDGMENT NEEDED]

INPUT (escalation rule):
If a pattern depends on knowing the course curriculum, student
demographics, or institutional context not in the data —
stop and say so. Do not interpret what you cannot know from data alone.
"""

print('✓ System prompt loaded — this shapes every step of the simulation')

---
## STEP 2 · The Model Plans

We pass the dataset summary and ask the model to write its analysis plan.

> **In a real agentic system** the model would decide to plan on its own.
> **In this simulation** we are prompting that step — but the content of the plan is the model's.

Watch: does it plan what you would plan?

In [ ]:
plan = ask_granite(
    system_prompt=SYSTEM,
    user_prompt=f"""Dataset: 943 students across 5 science courses.

Grade averages by subject (%):
{grade_by_subject.to_string()}

Write your analysis plan — what will you examine and in what order?""",
    label='STEP 2 · MODEL PLANS  [simulated — we prompted this step]'
)

---
## STEP 3 · The Model Analyzes

The model now uses its own plan as input. This is the **loop pattern** — previous output becomes next input.

> **In a real agentic system** the model would observe its plan and decide to analyze next.
> **In this simulation** we pass the output forward in code.

Watch: where does it decide what matters? Where does it write [HUMAN JUDGMENT NEEDED]?

In [ ]:
analysis = ask_granite(
    system_prompt=SYSTEM,
    user_prompt=f"""Your plan:
{plan}

Additional data:
Average time on platform by subject (hours):
{time_by_subject.to_string()}

Now execute your plan. Write:
1. Three key findings
2. One thing a human educator should examine more closely
Mark anything requiring human judgment: [HUMAN JUDGMENT NEEDED]""",
    label='STEP 3 · MODEL ANALYZES  [simulated — we passed the plan forward in code]'
)

---
## STEP 4 · The Model Names Its Own Limits

This is the escalation rule from the ICI prompt firing.

> **In a real agentic system** the model would recognize it had reached a boundary and stop.
> **In this simulation** we prompt it explicitly.

Watch: does it correctly identify where a human needed to step in?

In [ ]:
ask_granite(
    system_prompt=SYSTEM,
    user_prompt=f"""Your analysis:
{analysis}

Now reflect:
1. Name the exact point in your reasoning where you made an assumption
   that required knowing the curriculum, the students, or the institution.
2. What would a human educator need to know to make that call correctly?

Be specific. Name the exact moment.""",
    label='STEP 4 · MODEL NAMES ITS LIMITS  [escalation rule fires]'
)

---
## What Just Happened — The Simulated Loop

```
YOU WROTE THE ICI GOAL
         ↓
STEP 2 · Model planned      ← we prompted this step
         ↓  output passed forward in code
STEP 3 · Model analyzed     ← we prompted this step
         ↓  output passed forward in code
STEP 4 · Model reflected    ← we prompted this step
         ↓
         STOPPED — because the ICI boundary said to
```

**In the Extension session** — IBM BeeAI's RequirementAgent drives this loop itself.
You give it a goal. It decides what steps to take, calls tools, observes results,
and decides whether to continue or stop. You do not prompt each step.

**The architect's job is the same in both cases:**
Write the ICI goal that shapes what the loop can and cannot do.

---
### ✍️ Write your answer — double-click to edit

**The step I would stop the loop and require student judgment:**

*(Write here)*

**What the student would need to KNOW to make that decision:**

*(Write here)*

**How I would know they made it — and not the model:**

*(Write here)*

---
*Day 4 · Teaching with AI — Architects · Nagoya University · March 2026*
*Jeanne McClure, PhD · NC State Data Science & AI Academy · Ars Innovate Technologies and Consulting*
*NSF DUE-2313644*